Import Packages

In [2]:
import pandas as pd

Load the Data

In [3]:
df = pd.read_csv("WAHIS_data.csv")
print(df.head())

   Year      Semester World region  Country Administrative Division  \
0  2005  Jul-Dec 2005       Africa  Algeria                   Alger   
1  2005  Jul-Dec 2005       Africa  Algeria                   Alger   
2  2005  Jul-Dec 2005       Africa  Algeria                 Algeria   
3  2005  Jul-Dec 2005       Africa  Algeria                 Algeria   
4  2005  Jul-Dec 2005       Africa  Algeria                 Algeria   

                                             Disease  \
0  Varroa spp. (Inf. of honey bees with) (Varroosis)   
1  Varroa spp. (Inf. of honey bees with) (Varroosis)   
2                        Bovine tuberculosis (-2018)   
3                        Bovine tuberculosis (-2018)   
4                       Brucella abortus (Inf. with)   

  Serotype/Subtype/Genotype         Animal Category Event_id Species  \
0                       NaN  Both animal categories        -     NaN   
1                       NaN                Domestic        -    Bees   
2                   

/var/folders/9s/g3dm127d28j72x1bc2hpx0980000gp/T/ipykernel_1050/624074253.py:1: DtypeWarning: Columns (8,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("WAHIS_data.csv")


Clean and Filter Data

In [ ]:
# Filter for 2005 - 2024
df_filtered = df[df["Year"].between(2005, 2024)].copy()

# Identify orphan rows - species-detail rows without a summary row
group_cols = ["Year", "Semester", "World region", "Country", "Administrative Division", "Disease"]

is_smr = df_filtered["Outbreak_id"] == "-"

has_summary = (
    df_filtered.groupby(group_cols, dropna=False)["Animal Category"]
      .transform(lambda s: (s == "Both animal categories").any())
)
has_summary = has_summary.fillna(False).astype(bool)

orphan_mask = is_smr & ~has_summary
orphan_groups = df_filtered.loc[orphan_mask, group_cols].drop_duplicates()

print(f"{len(orphan_groups):,} orphan groups need a synthetic summary row")

# Build new rows shaped exactly like a real "Both animal categories" summary row
new_rows = orphan_groups.copy()
new_rows["Serotype/Subtype/Genotype"] = ""
new_rows["Animal Category"] = "Both animal categories"
new_rows["Species"] = ""
new_rows["Event_id"] = "-"
new_rows["Outbreak_id"] = "-"
new_rows["New outbreaks"] = 0 # Infer that those without summary row have 0 new outbreaks
new_rows["Susceptible"] = "-"
new_rows["Measuring units"] = "-"
new_rows["Cases"] = "-"
new_rows["Killed and disposed of"] = "-"
new_rows["Slaughtered"] = "-"
new_rows["Deaths"] = "-"
new_rows["Vaccinated"] = "-"

# Flag out synthetic rows so it's clear later which summary rows were real vs. inferred
df_filtered["is_synthetic_summary"] = False
new_rows["is_synthetic_summary"] = True

df_filtered = pd.concat([df_filtered, new_rows], ignore_index=True) # Use df_filtered if want to see species level data

print(f"Added {len(new_rows):,} synthetic summary rows. New total: {len(df_filtered):,}")

df_summary = df_filtered[df_filtered["Animal Category"] == "Both animal categories"] # Use df_summary to only see summary rows

82,775 orphan groups need a synthetic summary row
Added 82,775 synthetic summary rows. New total: 710,495
        Year      Semester World region      Country  \
710490  2024  Jul-Dec 2024      Oceania  New Zealand   
710491  2024  Jul-Dec 2024      Oceania  New Zealand   
710492  2024  Jul-Dec 2024      Oceania      Vanuatu   
710493  2024  Jul-Dec 2024      Oceania      Vanuatu   
710494  2024  Jul-Dec 2024      Oceania      Vanuatu   

                  Administrative Division  \
710490                                NaN   
710491                         Canterbury   
710492                              Sanma   
710493                              Shefa   
710494  Vanuatuan Exclusive Economic Zone   

                                                  Disease  \
710490  Paenibacillus larvae (Inf. of honey bees with)...   
710491  High pathogenicity avian influenza viruses (In...   
710492                  Bovine genital campylobacteriosis   
710493  Varroa spp. (Inf. of honey bees wi